# Deepfake Detection Project - Colab Notebook

Welcome! This Colab notebook runs the entire Deepfake Detection project end-to-end with Google Drive for storage and an optional small Celeb-DF subset for low disk usage.

## What you'll do here
- Mount Google Drive (storage for data/models/reports)
- Install project dependencies
- Prepare a small Celeb-DF subset (low-disk mode)
- Train EfficientNet-B0 on the subset
- Evaluate and generate reports (PDF/HTML)
- Run single-image inference with optional explainability (Grad-CAM)

## Architecture overview (short)
- File-based ML pipeline: images on disk → PyTorch datasets → models → metrics → reports
- Key modules:
  - Data: `src/data/preprocessing.py` (video → face crops), `src/data/dataloader.py` (PyTorch loaders)
  - Models: `src/models/baseline_models.py` (Xception, EfficientNet, ResNet, ViT), `src/models/advanced_models.py`
  - Training: `src/training/trainer.py` via `main.py train`
  - Evaluation: `src/evaluation/metrics.py` via `main.py evaluate`
  - Explainability: `src/explainability/gradcam.py`
  - Reporting: `src/reporting/report_generator.py` (PDF/HTML)
- Unified CLI: `main.py` supports train/evaluate/cross-dataset/inference

Tip: All artifacts will be saved to your Drive so they persist across sessions.



In [ ]:
# Mount Google Drive and set paths
from google.colab import drive
from pathlib import Path
import os

# 1) Mount Drive
drive.mount('/content/drive')

# 2) Set base paths (edit if you prefer a different folder)
PROJECT_DIR = Path('/content/Deepfake')
DRIVE_BASE = Path('/content/drive/MyDrive/deepfake_data')
RAW_DIR = DRIVE_BASE / 'raw'
PROCESSED_DIR = DRIVE_BASE / 'processed'

for p in [DRIVE_BASE, RAW_DIR, PROCESSED_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('Project dir:', PROJECT_DIR)
print('Drive base:', DRIVE_BASE)
print('Raw dir:', RAW_DIR)
print('Processed dir:', PROCESSED_DIR)


In [ ]:
# Get project code into Colab (choose one method)
import shutil, subprocess, sys

# Option A: Clone from a Git repo (recommended)
# !git clone https://github.com/<your-username>/Deepfake.git /content/Deepfake

# Option B: Copy from your Drive (upload your project there first)
# shutil.copytree('/content/drive/MyDrive/Deepfake', '/content/Deepfake', dirs_exist_ok=True)

# Verify structure
!ls -la /content | head -50



In [ ]:
# Install dependencies
%cd /content/Deepfake
%pip install -U pip
%pip install -r requirements.txt

# Quick import check
import sys
sys.path.insert(0, '/content/Deepfake/src')

try:
    import torch, numpy as np, cv2
    from src.utils.config import ConfigLoader
    from src.models.baseline_models import create_model
    print('✅ Imports OK:', torch.__version__)
except Exception as e:
    print('❌ Import error:', e)


## Prepare a small Celeb-DF subset (low disk)

Place a few videos per class in Drive:
- `/content/drive/MyDrive/deepfake_data/raw/celebd/Celeb-real/*.mp4`
- `/content/drive/MyDrive/deepfake_data/raw/celebd/Celeb-synthesis/*.mp4`

Then run the cell below to preprocess a tiny subset.


In [ ]:
# Preprocess Celeb-DF subset (low-disk mode)
%cd /content/Deepfake

!python src/data/preprocessing.py \
  --dataset celebd \
  --data_path /content/drive/MyDrive/deepfake_data/raw/celebd \
  --output_path /content/drive/MyDrive/deepfake_data/processed/celebd_5k_fast \
  --face_detector opencv \
  --max_videos_per_class 2500 \
  --frame_stride 90 \
  --max_frames_per_video 2


In [ ]:
# Train EfficientNet-B0 on the subset
%cd /content/Deepfake

!python main.py train \
  --model efficientnet_b0 \
  --dataset celebd_5k_fast \
  --data_root /content/drive/MyDrive/deepfake_data/processed \
  --epochs 10 \
  --batch_size 16


In [ ]:
# Evaluate the trained model (latest)
%cd /content/Deepfake

import glob, os
latest = sorted(glob.glob('/content/drive/MyDrive/deepfake_data/processed/experiments/*/final_model.pth'))
if latest:
    model_path = latest[-1]
    print('Using model:', model_path)
    !python main.py evaluate \
      --model_path "$model_path" \
      --dataset celebd_mini \
      --data_root /content/drive/MyDrive/deepfake_data/processed \
      --batch_size 16
else:
    print('No model found in Drive experiments folder.')


In [ ]:
# Inference on a single image (with optional explainability)
%cd /content/Deepfake

# Pick an image from the processed set (or upload your own to Drive)
import glob
images = glob.glob('/content/drive/MyDrive/deepfake_data/processed/celebd_mini/images/*.jpg')
if images:
    test_image = images[0]
    latest = sorted(glob.glob('/content/drive/MyDrive/deepfake_data/processed/experiments/*/final_model.pth'))
    model_path = latest[-1] if latest else ''
    print('Testing image:', test_image)
    print('Using model:', model_path)
    !python main.py inference \
      --model_path "$model_path" \
      --image_path "$test_image" \
      --explainability
else:
    print('No images found. Run preprocessing or upload an image to Drive.')


## Reports & artifacts
- Training experiments: `/content/drive/MyDrive/deepfake_data/processed/experiments/`
- Reports (PDF/HTML): `/content/drive/MyDrive/deepfake_data/processed/reports/`
- Explainability images: `/content/drive/MyDrive/deepfake_data/processed/reports/explainability/`

Open the HTML report in your browser (download from Drive):
- `*_report.html` contains metrics, curves, and summaries.
